# Notebook 1 — Read and Join Data

## Objective

- Connect to the PostgreSQL database.
- Read the required Olist tables.
- Inspect the structure and basic characteristics of the data.
- Understand primary keys, foreign keys, duplicates, and table grain.
- Join the required tables.
- Create a final ML-ready table with one row per order.

In [4]:
import pandas as pd

import psycopg2


In [5]:
from getpass import getpass
password = getpass("postgresSQL password:")

conn = psycopg2.connect(
    dbname="Brazilian E-Commerce",
    user="postgres",
    password=password,
    host="127.0.0.1",
    port=5432
)

print("Connection successful!")

Connection successful!


In [6]:
cursor = conn.cursor()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

tables = [row[0] for row in cursor.fetchall()]

print(tables)

['olist_customers', 'olist_geolocation', 'olist_order_items', 'olist_order_payments', 'olist_order_reviews', 'olist_orders', 'olist_products', 'olist_sellers', 'product_category_name_translation']


In [7]:
orders = pd.read_sql("SELECT * FROM olist_orders", conn)

customers = pd.read_sql("SELECT * FROM olist_customers", conn)

order_items = pd.read_sql("SELECT * FROM olist_order_items", conn)

order_payments = pd.read_sql("SELECT * FROM olist_order_payments", conn)

order_reviews = pd.read_sql("SELECT * FROM olist_order_reviews", conn)

products = pd.read_sql("SELECT * FROM olist_products", conn)

sellers = pd.read_sql("SELECT * FROM olist_sellers", conn)

geolocation = pd.read_sql("SELECT * FROM olist_geolocation", conn)

category_translation = pd.read_sql(
    "SELECT * FROM product_category_name_translation",
    conn
)

C:\Users\shima\AppData\Local\Temp\ipykernel_22644\3438964855.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql("SELECT * FROM olist_orders", conn)
C:\Users\shima\AppData\Local\Temp\ipykernel_22644\3438964855.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql("SELECT * FROM olist_customers", conn)
C:\Users\shima\AppData\Local\Temp\ipykernel_22644\3438964855.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql("SELECT * FROM olist_order_items", conn)
C:\

In [8]:
MlTable=orders.copy()

In [9]:
MlTable = MlTable.merge(customers, on='customer_id', how='left')    

In [10]:
reviews_agg = (
    order_reviews.groupby("order_id")
    .agg(
        {
            "review_score": "mean",
            "review_id": "count",
            "review_comment_message": lambda x: (
                " | ".join(x.dropna()) if not x.dropna().empty else None
            ),
            "review_creation_date": "max",
        }
    )
    .reset_index()
)
reviews_agg = reviews_agg.rename(
    columns={
        "review_score": "avg_review_score",  # متوسط التقييم
        "review_id": "review_count",  # عدد المراجعات (بدلاً من الـ ID الوهمي)
        "review_comment_message": "combined_review_comments",  # التعليقات المدمجة
        "review_creation_date": "latest_review_date",  # أحدث تاريخ مراجعة
    }
)
MlTable = MlTable.merge(reviews_agg, on="order_id", how="left")

In [11]:
order_payments_agg= (
    order_payments.groupby("order_id").agg(
        {
            "payment_value": "sum",
            "payment_type":( lambda x: ", ".join(x.dropna().unique())),
            "payment_sequential": "count",
            "payment_installments": "max",

        }

    ).reset_index()
)# إعادة تسمية أعمدة المدفوعات لتكون أكثر دقة ووصفاً لمحتواها الجديد
order_payments_agg = order_payments_agg.rename(
    columns={
        "payment_sequential": "payment_count",  # عدد حركات/طرق الدفع (بعد العد)
        "payment_installments": "max_payment_installments",  # أقصى عدد أقساط
    }
)

MlTable = MlTable.merge(order_payments_agg, on="order_id", how="left")



In [12]:
# 1. حذف النسخ الزائدة التي تحمل اللاحقة _y فقط والإبقاء على _x
MlTable = MlTable.drop(
    columns=[
        "payment_value_y",
        "payment_type_y",
    ],
    errors="ignore",
)

# 2. إعادة تسمية أعمدة _x لتصبح بالاسم النهائي النظيف
MlTable = MlTable.rename(
    columns={
        "payment_value_x": "payment_value",
        "payment_type_x": "payment_type",
    }
)


In [13]:
MlTable.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'avg_review_score', 'review_count',
       'combined_review_comments', 'latest_review_date', 'payment_value',
       'payment_type', 'payment_count', 'max_payment_installments'],
      dtype='str')

In [14]:

order_items_agg = (
    order_items.groupby("order_id")
    .agg(
        {
            "order_item_id": "count",  
            "price": "sum", 
            "freight_value": "sum", 
            "shipping_limit_date": "max", 
        }
    )
    .reset_index()
)


order_items_agg = order_items_agg.rename(
    columns={
        "order_item_id": "total_items_count",
        "price": "total_items_price",
        "freight_value": "total_freight_value",
    }
)


MlTable = MlTable.merge(order_items_agg, on="order_id", how="left")

In [15]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [16]:
# 1. ربط جدول عناصر الطلب مع جدول المنتجات مؤقتاً
items_with_products = order_items.merge(products, on="product_id", how="left")

# 2. تلخيص خصائص المنتجات وإضافة عدد الصور على مستوى الطلب الواحد (order_id)
order_products_agg = (
    items_with_products.groupby("order_id")
    .agg(
        {
            "product_weight_g": "sum",  # إجمالي وزن المنتجات في الطلب
            "product_length_cm": "max",  # أقصى طول لعنصر في الطلب
            "product_height_cm": "max",  # أقصى ارتفاع لعنصر في الطلب
            "product_width_cm": "max",  # أقصى عرض لعنصر في الطلب
            "product_photos_qty": "sum",  # إجمالي عدد الصور للمنتجات في الطلب
            "product_category_name": (
                lambda x: x.mode()[0] if not x.mode().empty else "unknown"
            ),  # الفئة الأكثر تكراراً في الطلب
        }
    )
    .reset_index()
)

# 3. إعادة تسمية الأعمدة لتكون واضحة
order_products_agg = order_products_agg.rename(
    columns={
        "product_weight_g": "total_product_weight_g",
        "product_photos_qty": "total_product_photos",
        "product_category_name": "primary_product_category",
    }
)

# 4. دمج الملخص النهائي مع الـ MlTable
MlTable = MlTable.merge(order_products_agg, on="order_id", how="left")

In [17]:
items_with_sellers = order_items.merge(sellers, on="seller_id", how="left")

order_sellers_agg = (
    items_with_sellers.groupby("order_id")
    .agg(
        {
            "seller_id": "nunique",  # عدد البائعين المختلفين في نفس الطلب
            "seller_zip_code_prefix": lambda x: ", ".join(
                x.dropna().astype(str).unique()
            ),  # الرموز البريدية مجتمعة
            "seller_city": lambda x: ", ".join(
                x.dropna().astype(str).unique()
            ),  # مدن البائعين مجتمعة
            "seller_state": lambda x: ", ".join(
                x.dropna().astype(str).unique()
            ),  # ولايات البائعين مجتمعة
        }
    )
    .reset_index()
)


order_sellers_agg = order_sellers_agg.rename(
    columns={
        "seller_id": "total_unique_sellers",
        "seller_zip_code_prefix": "sellers_zip_codes_combined",
        "seller_city": "sellers_cities_combined",
        "seller_state": "sellers_states_combined",
    }
)


MlTable = MlTable.merge(order_sellers_agg, on="order_id", how="left")

In [18]:
# 1. حذف الأعمدة المنتهية بـ _x (القديمة أو المتكررة الزائدة)
columns_to_drop = [col for col in MlTable.columns if col.endswith("_x")]
MlTable = MlTable.drop(columns=columns_to_drop)

# 2. إعادة تسمية الأعمدة المنتهية بـ _y لتزيل عنها اللاحقة وتصبح باسمها الصحيح
rename_dict = {
    col: col[:-2] for col in MlTable.columns if col.endswith("_y")
}
MlTable = MlTable.rename(columns=rename_dict)

In [19]:
import numpy as np


# 1. تلخيص جدول الجغرافيا (نقطة واحدة لكل رمز بريدي وتوحيد نوع البيانات لنص)
geo_agg = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg({"geolocation_lat": "mean", "geolocation_lng": "mean"})
    .reset_index()
)
geo_agg["geolocation_zip_code_prefix"] = (
    geo_agg["geolocation_zip_code_prefix"].astype(str).str.strip()
)

# 2. جلب إحداثيات العميل بعد التأكد من نوع البيانات
MlTable["customer_zip_code_prefix"] = (
    MlTable["customer_zip_code_prefix"].astype(str).str.strip()
)
MlTable = MlTable.merge(
    geo_agg.rename(
        columns={
            "geolocation_zip_code_prefix": "customer_zip_code_prefix",
            "geolocation_lat": "customer_lat",
            "geolocation_lng": "customer_lng",
        }
    ),
    on="customer_zip_code_prefix",
    how="left",
)

# 3. استخراج الرمز البريدي للبائع الرئيسي وتوحيد نوع البيانات لنص
MlTable["primary_seller_zip"] = (
    MlTable["sellers_zip_codes_combined"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.strip()
)

MlTable = MlTable.merge(
    geo_agg.rename(
        columns={
            "geolocation_zip_code_prefix": "primary_seller_zip",
            "geolocation_lat": "seller_lat",
            "geolocation_lng": "seller_lng",
        }
    ),
    on="primary_seller_zip",
    how="left",
)


# 4. دالة حساب المسافة بالكيلومترات (Haversine Formula)
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
  lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
  dlon = lon2 - lon1
  dlat = lat2 - lat1
  a = (
      np.sin(dlat / 2.0) ** 2
      + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
  )
  c = 2 * np.arcsin(np.sqrt(a))
  km = 6367 * c  # نصف قطر الأرض بالكيلومتر
  return km




In [20]:


MlTable["distance_km"] = calculate_haversine_distance(
    MlTable["customer_lat"],
    MlTable["customer_lng"],
    MlTable["seller_lat"],
    MlTable["seller_lng"],
)

In [21]:
MlTable.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'avg_review_score', 'review_count',
       'combined_review_comments', 'latest_review_date', 'payment_value',
       'payment_type', 'payment_count', 'max_payment_installments',
       'total_items_count', 'total_items_price', 'total_freight_value',
       'shipping_limit_date', 'total_product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'total_product_photos',
       'primary_product_category', 'total_unique_sellers',
       'sellers_zip_codes_combined', 'sellers_cities_combined',
       'sellers_states_combined', 'customer_lat', 'customer_lng',
       'primary_seller_zip', 'seller_lat', 'seller_lng', 'distance_km'],
      dtype='str')

In [22]:
from pathlib import Path

Path("artifacts").mkdir(exist_ok=True)

MlTable.to_parquet("artifacts/ml_table.parquet")

In [23]:
import pandas as pd
import pyarrow as pa

print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)

pandas: 3.0.5
pyarrow: 25.0.1
